In [24]:
import pandas as pd
matches = pd.read_csv(
    "../../data/processed/match_data/matches_with_elo.csv"
)

fifa = pd.read_csv(
    "../../data/processed/team_data/fifa_yearly_rankings.csv"
)
matches["date"] = pd.to_datetime(matches["date"])

matches["year"] = matches["date"].dt.year
home_fifa = fifa.rename(
    columns={
        "team": "home_team",
        "rank": "home_rank",
        "total.points": "home_points"
    }
)
home_fifa = home_fifa[
    ["date", "home_team", "home_rank", "home_points"]
]
matches = matches.merge(
    home_fifa,
    left_on=["year", "home_team"],
    right_on=["date", "home_team"],
    how="left"
)

away_fifa = fifa.rename(
    columns={
        "team": "away_team",
        "rank": "away_rank",
        "total.points": "away_points"
    }
)

away_fifa = away_fifa[
    ["date", "away_team", "away_rank", "away_points"]
]

matches = matches.merge(
    away_fifa,
    left_on=["year", "away_team"],
    right_on=["date", "away_team"],
    how="left"
)


matches["rank_diff"] = (
    matches["home_rank"]
    - matches["away_rank"]
)

matches["points_diff"] = (
    matches["home_points"]
    - matches["away_points"]
)


latest_fifa = (
    fifa.sort_values("date")
        .groupby("team")
        .tail(1)
)
latest_rank = dict(
    zip(latest_fifa["team"],
        latest_fifa["rank"])
)

latest_points = dict(
    zip(latest_fifa["team"],
        latest_fifa["total.points"])
)
matches["home_rank"] = (
    matches["home_rank"]
    .fillna(
        matches["home_team"].map(latest_rank)
    )
)

matches["home_points"] = (
    matches["home_points"]
    .fillna(
        matches["home_team"].map(latest_points)
    )
)
matches["away_rank"] = (
    matches["away_rank"]
    .fillna(
        matches["away_team"].map(latest_rank)
    )
)

matches["away_points"] = (
    matches["away_points"]
    .fillna(
        matches["away_team"].map(latest_points)
    )
)

matches["rank_diff"] = (
    matches["home_rank"]
    - matches["away_rank"]
)

matches["points_diff"] = (
    matches["home_points"]
    - matches["away_points"]
)

matches = matches.drop(
    columns=[
        "date_y",
        "date",
        "year",
    ]
)
matches = matches.rename(
    columns={
        "date_x":"date"
    }
)

matches= matches.drop(columns=['city', 'country'])
matches.columns
matches.to_csv(
    "../../data/processed/match_data/matches_with_elo_fifa.csv",
    index=False
)

home_rank      1324
away_rank      1316
home_points    1324
away_points    1316
dtype: int64
home_rank      0
away_rank      0
home_points    0
away_points    0
dtype: int64
